In [1]:
!pip install -q openai

In [2]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Why did the scarecrow become a successful motivational speaker?

Because he was outstanding in his field—and he always knew how to **stay grounded** while still raising everyone's spirits! 😄


In [3]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We’d love to make an eggless cake for you, but we need at least 24 hours’ notice, so today evening isn’t possible. Please place the order for tomorrow and tell us the flavor you’d like!
CompletionUsage(completion_tokens=188, prompt_tokens=183, total_tokens=371, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=133, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.142090532, prompt_time=0.05740806, completion_time=0.403760787, total_time=0.461168847)


In [4]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}   # key → value

In [19]:
#MENU["donut"]                  # 500
MENU.get("donut", "donuts not available")   # 'not on menu'  (no crash)
#MENU["donut"]

'donuts not available'

In [7]:
order = {"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": True}


In [8]:
order["qty"] * MENU[order["item"]] # 1000

1000

In [9]:
import json
text = json.dumps(order)           # dict → JSON string, to send or save
back = json.loads(text)

In [10]:
text

'{"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": true}'

In [11]:
back

{'name': 'Ravi', 'item': 'plum cake', 'qty': 2, 'eggless': True}

In [12]:
  def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [13]:
def get_price(item, qty=1) :
    """Return the total price in rupees for qty of an item on the Sweet Crumbs menu."""
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [14]:
a=90
print(a+b)
print('hi')

NameError: name 'b' is not defined

In [15]:
try:
  a=10
  print(a+b)
  print('hi')
except:
  print('b is not defined')


b is not defined


In [20]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: hi
Bot: Hi! 👋 Welcome to Sweet Crumbs Home Bakery. How can I help you today? We have plum cake (₹500), chocolate cake (₹650) and cupcakes (₹80). Let me know what you’d like!
You: plum cake
Bot: Great choice! Plum cake is ₹500 each. Would you like eggless? How many and your name, please? Please note we need 24‑hour notice.
You: ok
Bot: Sure! How many plum cakes would you like, and may I have your name for the order? (We need 24‑hour notice.)
You: 8
Bot: Thanks! 8 plum cakes will be ₹4,000. Would you like the eggless version, and could you share your name for the order? (We need 24‑hour notice.)
You: naveen
Bot: Naveen, 8 plum cakes will be ₹4,000. Would you like the eggless version? Please confirm your contact number.
You: with egg and 9876543210
Bot: Naveen, 8 plum cakes (with egg) total ₹4,000. Got your contact 9876543210. We’ll need 24‑hour notice to prepare. Let me know if everything looks good!
You: thanks
Bot: You’re all set, Naveen! 8 plum cakes with egg, total ₹4,000, conta

In [21]:
history

[{'role': 'system',
  'content': 'ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.\nFACTS:  Menu with prices in rupees: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n        Orders need 24 hours\' notice. All cakes have eggless versions.\nRULES:  Never promise a delivery time. If it isn\'t in FACTS, say "I\'ll check with the owner."\n        Remember the customer\'s name and order during the chat.\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'hi'},
 {'role': 'assistant',
  'content': 'Hi! 👋 Welcome to Sweet Crumbs Home Bakery. How can I help you today? We have plum cake (₹500), chocolate cake (₹650) and cupcakes (₹80). Let me know what you’d like!'},
 {'role': 'user', 'content': 'plum cake'},
 {'role': 'assistant',
  'content': 'Great choice! Plum cake is ₹500 each. Would you like eggless? How many and your name, please? Please note we need 24‑hour notice.'},
 {'role': 'user', 'content': 'ok'